In [1]:
#required for manipulating data
import pandas as pd
import numpy as np 

#required for building the interactive dashboard
import panel as pn
pn.extension('tabulator')
import hvplot.pandas
import holoviews as hv
hv.extension('bokeh')

In [2]:
# import csv file

df = pd.read_csv('data/transactions_2024.csv')

In [3]:
df

,Date,Name / Description,Expense/Income,Amount (Argentinian Peso)
0,19/01/24,TRANSF. CLIENTE ESPOSITO MARCELO,Income,2000.00
1,19/01/24,TRANSFERENCIA INMEDIATA,Income,980.00
2,22/01/24,IMP PAIS SD,Expense,-130.49
3,22/01/24,APPLE.COM/BILL,Expense,-1731.30
4,22/01/24,RG 4815/20,Expense,-489.36
...,...,...,...,...
120,26/08/24,TRANSFERENCIA,Expense,-30000.00
121,26/08/24,DIA TIENDA 679,Expense,-5436.25
122,27/08/24,VERDULERIA PAOLA,Expense,-7346.00
123,28/08/24,TRANSF. CLIENTE ESPOSITO MARCELO,Income,30000.00


In [4]:
#clean df

df = df.rename(columns={'Name / Description': 'Description'})   #rename columns
df['Description'] = df['Description'].map(str.lower) #lower case of descriptions
df['Category'] = 'unassigned'                        #add category column

df.head()


,Date,Description,Expense/Income,Amount (Argentinian Peso),Category
0,19/01/24,transf. cliente esposito marcelo,Income,2000.00,unassigned
1,19/01/24,transferencia inmediata,Income,980.00,unassigned
2,22/01/24,imp pais sd,Expense,-130.49,unassigned
3,22/01/24,apple.com/bill,Expense,-1731.30,unassigned
4,22/01/24,rg 4815/20,Expense,-489.36,unassigned


In [5]:
#define all categories

# Bank Transfers
# Tax & Fees
# Digital Services
# Services
# Groceries
# Food & Dining
# Shopping
# Health & Pharmacy
# Interest
# Reimbursement




In [6]:
#Assign transactions to the correct category

# Bank Transfer
df['Category'] = np.where(df['Description'].str.contains(
    'transf|transferencia'), 
    'Bank Transfer', df['Category'])

# Tax & Fees
df['Category'] = np.where(df['Description'].str.contains(
    'imp pais|rg 4815|iva serv digit'), 
    'Tax & Fees', df['Category'])

# Digital Services
df['Category'] = np.where(df['Description'].str.contains(
    'apple.com'), 
    'Digital Services', df['Category'])

# Services
df['Category'] = np.where(df['Description'].str.contains(
    'openpay|merpago|julio cesar estrada|juanlin|lanaturaleza1855'), 
    'Services', df['Category'])

# Groceries
df['Category'] = np.where(df['Description'].str.contains(
    'dia tienda|coto sucursal|pvs*super uruburu jose e|verduleria paola|market avenida'), 
    'Groceries', df['Category'])

# Food & Dining
df['Category'] = np.where(df['Description'].str.contains(
    'fei li|la finca'), 
    'Food & Dining', df['Category'])

# Shopping
df['Category'] = np.where(df['Description'].str.contains(
    'pigmento|new garden'), 
    'Shopping', df['Category'])

# Health & Pharmacy
df['Category'] = np.where(df['Description'].str.contains(
    'farmacity'), 
    'Health & Pharmacy', df['Category'])

# Interest
df['Category'] = np.where(df['Description'].str.contains(
    'intereses ganados'), 
    'Interest', df['Category'])

# Reimbursement
df['Category'] = np.where(df['Description'].str.contains(
    'reintegro modo'), 
    'Reimbursement', df['Category'])

# Convert the "Date" column to a datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%y')

# Extract the month and year information
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

pd.options.display.max_rows = 999
df.head(200)

,Date,Description,Expense/Income,Amount (Argentinian Peso),Category,Month,Year
0,2024-01-19,transf. cliente esposito marcelo,Income,2000.00,Bank Transfer,1,2024
1,2024-01-19,transferencia inmediata,Income,980.00,Bank Transfer,1,2024
2,2024-01-22,imp pais sd,Expense,-130.49,Tax & Fees,1,2024
3,2024-01-22,apple.com/bill,Expense,-1731.30,Digital Services,1,2024
4,2024-01-22,rg 4815/20,Expense,-489.36,Tax & Fees,1,2024
5,2024-01-23,iva serv digit-rg afip 4240,Expense,-363.57,Tax & Fees,1,2024
6,2024-02-28,transf. cliente esposito marcelo,Income,30000.00,Bank Transfer,2,2024
7,2024-02-29,openpay*vida point,Expense,-2900.00,Services,2,2024
8,2024-02-29,dia tienda 268,Expense,-3518.75,Groceries,2,2024
9,2024-02-29,fei li,Expense,-3000.00,Food & Dining,2,2024


In [7]:
#check unassigned transactions and confirm all transactions are assigned to a category

unassigned = df.loc[df['Category'] == 'unassigned']
unassigned

,Date,Description,Expense/Income,Amount (Argentinian Peso),Category,Month,Year
17,2024-03-05,pvs*super uruburu jose e,Expense,-7500.0,unassigned,3,2024
41,2024-03-18,pvs*super uruburu jose e,Expense,-6750.0,unassigned,3,2024
51,2024-04-03,pvs*super uruburu jose e,Expense,-7280.0,unassigned,4,2024


## Create Top Banner for a summary of last month's income, recurring expenses, non-recurring expenses and savings


In [8]:
# Get the latest month and year
latest_month = df['Month'].max()
latest_year = df['Year'].max()

# Filter the dataframe to include only transactions from the latest month
last_month_expenses = df[(df['Month'] == latest_month) & (df['Year'] == latest_year)]

In [9]:
last_month_expenses = last_month_expenses.groupby('Category')['Amount (Argentinian Peso)'].sum().reset_index()

last_month_expenses['Amount (Argentinian Peso)']=last_month_expenses['Amount (Argentinian Peso)'].astype('str')
last_month_expenses['Amount (Argentinian Peso)']=last_month_expenses['Amount (Argentinian Peso)'].str.replace('-','')
last_month_expenses['Amount (Argentinian Peso)']=last_month_expenses['Amount (Argentinian Peso)'].astype('float')        #get absolute figures

last_month_expenses = last_month_expenses[last_month_expenses["Category"].str.contains("unassigned") == False]    #exclude "unassigned" category
last_month_expenses = last_month_expenses.sort_values(by='Amount (Argentinian Peso)', ascending=False)    #sort values
last_month_expenses['Amount (Argentinian Peso)'] = last_month_expenses['Amount (Argentinian Peso)'].round().astype(int)      #round values

last_month_expenses

,Category,Amount (Argentinian Peso)
0,Bank Transfer,70000
2,Groceries,33636
1,Digital Services,2972
4,Tax & Fees,1691
3,Interest,1


In [10]:
last_month_expenses_tot = last_month_expenses['Amount (Argentinian Peso)'].sum()
last_month_expenses_tot

np.int64(108300)

In [11]:
def calculate_difference(event):
    income = float(income_widget.value)
    recurring_expenses = float(recurring_expenses_widget.value)
    monthly_expenses = float(monthly_expenses_widget.value)
    difference = income - recurring_expenses - monthly_expenses
    difference_widget.value = str(difference)

income_widget = pn.widgets.TextInput(name="Income", value="0")
recurring_expenses_widget = pn.widgets.TextInput(name="Recurring Expenses", value="0")
monthly_expenses_widget = pn.widgets.TextInput(name="Non-Recurring Expenses", value=str(last_month_expenses_tot))
difference_widget = pn.widgets.TextInput(name="Last Month's Savings", value="0")

income_widget.param.watch(calculate_difference, "value")
recurring_expenses_widget.param.watch(calculate_difference, "value")
monthly_expenses_widget.param.watch(calculate_difference, "value")

pn.Row(income_widget, recurring_expenses_widget, monthly_expenses_widget, difference_widget).show()


Launching server at http://localhost:64593


## Create last month expenses bar chart 

In [12]:
last_month_expenses_chart = last_month_expenses.hvplot.bar(
    x='Category', 
    y='Amount (Argentinian Peso)', 
    height=250, 
    width=850, 
    title="Last Month Expenses",
    ylim=(0, 500))

last_month_expenses_chart

:Bars   [Category]   (Amount (Argentinian Peso))

## Create monthly expenses trend bar chart 

In [13]:
df['Date'] = pd.to_datetime(df['Date'])            # convert the 'Date' column to a datetime object
df['Month-Year'] = df['Date'].dt.to_period('M')    # extract the month and year from the 'Date' column and create a new column 'Month-Year'
monthly_expenses_trend_by_cat = df.groupby(['Month-Year', 'Category'])['Amount (Argentinian Peso)'].sum().reset_index()

monthly_expenses_trend_by_cat['Amount (Argentinian Peso)']=monthly_expenses_trend_by_cat['Amount (Argentinian Peso)'].astype('str')
monthly_expenses_trend_by_cat['Amount (Argentinian Peso)']=monthly_expenses_trend_by_cat['Amount (Argentinian Peso)'].str.replace('-','')
monthly_expenses_trend_by_cat['Amount (Argentinian Peso)']=monthly_expenses_trend_by_cat['Amount (Argentinian Peso)'].astype('float')
monthly_expenses_trend_by_cat = monthly_expenses_trend_by_cat[monthly_expenses_trend_by_cat["Category"].str.contains("unassigned") == False]

monthly_expenses_trend_by_cat = monthly_expenses_trend_by_cat.sort_values(by='Amount (Argentinian Peso)', ascending=False)
monthly_expenses_trend_by_cat['Amount (Argentinian Peso)'] = monthly_expenses_trend_by_cat['Amount (Argentinian Peso)'].round().astype(int)
monthly_expenses_trend_by_cat['Month-Year'] = monthly_expenses_trend_by_cat['Month-Year'].astype(str)
monthly_expenses_trend_by_cat = monthly_expenses_trend_by_cat.rename(columns={'Amount (Argentinian Peso)':'Amount (Argentinian Peso)'})

monthly_expenses_trend_by_cat

,Month-Year,Category,Amount (Argentinian Peso)
7,2024-03,Bank Transfer,110000
23,2024-05,Bank Transfer,100000
32,2024-07,Bank Transfer,94688
16,2024-04,Bank Transfer,80600
34,2024-07,Groceries,72278
10,2024-03,Groceries,70120
40,2024-08,Bank Transfer,70000
18,2024-04,Groceries,41744
25,2024-05,Groceries,36632
42,2024-08,Groceries,33636


In [14]:
#Define Panel widget

select_category1 = pn.widgets.Select(name='Select Category', options=[
    'All',
    'Bank Transfer',
    'Tax & Fees',
    'Digital Services',
    'Services',
    'Groceries',
    'Food & Dining',
    'Shopping',
    'Health & Pharmacy',
    'Interest',
    'Reimbursement',
])

select_category1

BokehModel(combine_events=True, render_bundle={'docs_json': {'abbb6401-c530-4d6e-bd0e-d3d84ad29179': {'version…

In [15]:
# define plot function
def plot_expenses(category):
    if category == 'All':
        plot_df = monthly_expenses_trend_by_cat.groupby('Month-Year').sum()
    else:
        plot_df = monthly_expenses_trend_by_cat[monthly_expenses_trend_by_cat['Category'] == category].groupby('Month-Year').sum()
    plot = plot_df.hvplot.bar(x='Month-Year', y='Amount (Argentinian Peso)')
    return plot

# define callback function
@pn.depends(select_category1.param.value)
def update_plot(category):
    plot = plot_expenses(category)
    return plot

# create layout
monthly_expenses_trend_by_cat_chart = pn.Row(select_category1, update_plot)
monthly_expenses_trend_by_cat_chart[1].width = 600

monthly_expenses_trend_by_cat_chart

BokehModel(combine_events=True, render_bundle={'docs_json': {'4fc62060-1da2-4666-a73a-d1a417cba2a3': {'version…

## Create summary table

In [16]:
df = df[['Date', 'Category', 'Description', 'Amount (Argentinian Peso)']]
df['Amount (Argentinian Peso)']=df['Amount (Argentinian Peso)'].astype('str')
df['Amount (Argentinian Peso)']=df['Amount (Argentinian Peso)'].str.replace('-','')
df['Amount (Argentinian Peso)']=df['Amount (Argentinian Peso)'].astype('float')        #get absolute figures

df = df[df["Category"].str.contains("unassigned") == False]    #exclude "unassigned" category
df['Amount (Argentinian Peso)'] = df['Amount (Argentinian Peso)'].round().astype(int)      #round values
df


,Date,Category,Description,Amount (Argentinian Peso)
0,2024-01-19,Bank Transfer,transf. cliente esposito marcelo,2000
1,2024-01-19,Bank Transfer,transferencia inmediata,980
2,2024-01-22,Tax & Fees,imp pais sd,130
3,2024-01-22,Digital Services,apple.com/bill,1731
4,2024-01-22,Tax & Fees,rg 4815/20,489
5,2024-01-23,Tax & Fees,iva serv digit-rg afip 4240,364
6,2024-02-28,Bank Transfer,transf. cliente esposito marcelo,30000
7,2024-02-29,Services,openpay*vida point,2900
8,2024-02-29,Groceries,dia tienda 268,3519
9,2024-02-29,Food & Dining,fei li,3000


In [17]:
# Define a function to filter the dataframe based on the selected category
def filter_df(category):
    if category == 'All':
        return df
    return df[df['Category'] == category]
# Create a DataFrame widget that updates based on the category filter
summary_table = pn.widgets.DataFrame(filter_df('All'), height = 300,width=400)

# Define a callback that updates the dataframe widget when the category filter is changed
def update_summary_table(event):
    summary_table.value = filter_df(event.new)

# Add the callback function to the category widget
select_category1.param.watch(update_summary_table, 'value')

summary_table

BokehModel(combine_events=True, render_bundle={'docs_json': {'29fde988-9848-4a9e-b0a6-27659c891938': {'version…

## Create Final Dashboard

In [18]:
template = pn.template.FastListTemplate(
    title="Personal Finances Summary",
    sidebar=[
        pn.pane.Markdown("## *If you can't manage your money, making more won't help*"),
        pn.pane.PNG('finance.png', sizing_mode='scale_both'),
        pn.pane.Markdown(""),
        pn.pane.Markdown(""),
        select_category1
    ],
    main=[
        pn.Row(income_widget, recurring_expenses_widget, monthly_expenses_widget, difference_widget, width=950),
        pn.Row(last_month_expenses_chart, height=240),
        pn.GridBox(
            monthly_expenses_trend_by_cat_chart[1],
            summary_table,
            ncols=2,
            width=500,  
            align='start',
            sizing_mode='stretch_width'
        )
    ]
)

template.show()


Launching server at http://localhost:64601


2025-01-12 21:39:16,247 ERROR: panel.reactive - Callback failed for object named 'Income' changing property {'value': ''} 
Traceback (most recent call last):
  File "C:\Users\esposito\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\panel\reactive.py", line 470, in _process_events
    self.param.update(**self_params)
  File "C:\Users\esposito\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\param\parameterized.py", line 2406, in update
    restore = dict(self_._update(arg, **kwargs))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\esposito\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\param\parameterized.py", line 2439, in _update
    self_._batch_call_watchers()
  File "C:\Users\esposito\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.